# PromptAlign: full-data QA from repository scripts
CPU only; Internet on. Run All. This does not train or rerun CLIP.
Cell 2 defaults to downloading ~4.76 GB on Kaggle. Executing with DOWNLOAD_DATASET=True opts in. To reuse all 12 mounted shards, set it False and fill DATA_ROOT.
Send back data-qa-*.zip and this notebook with outputs. REVIEW_REQUIRED is expected before training.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hoan25/BaiqiL-GenAI-Bench.git"
GIT_REF = "main"
PRIVATE_REPO = False
ROOT = Path("/kaggle/working/promptalign-ai")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
if PRIVATE_REPO:
    import base64
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authorization = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + authorization)
try:
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], env=env, check=True)
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=ROOT, text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Existing directory belongs to a different repository")
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip():
        raise RuntimeError("Working tree has changes; preserve them before changing revision")
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=ROOT, env=env, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
finally:
    for key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
        env.pop(key, None)
    if PRIVATE_REPO:
        del token, authorization
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())


In [ ]:
DOWNLOAD_DATASET = True  # explicit opt-in when you execute this cell
DATA_ROOT = "/kaggle/input/genai-bench"  # used only when download is False
command = [sys.executable, "-u", str(ROOT / "code/scripts/kaggle_data.py")]
command += ["--download-dataset"] if DOWNLOAD_DATASET else ["--data-root", DATA_ROOT]
result = subprocess.run(command, cwd=ROOT / "code")
print("Download the ZIP from Kaggle Output / Files under /kaggle/working.")
print("These are filesystem paths, not browser URLs:")
for archive in sorted(Path("/kaggle/working").glob("data-qa-*.zip")):
    print(f"  {archive.name} ({archive.stat().st_size:,} bytes)")
print("Exit code:", result.returncode)
